# 관통 프로젝트 4단계 (풀이)

검증용 풀이 노트북입니다. 공유하지 않습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 실행합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(1장의 공개 LLM, 임베딩 모델)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정해 두면 모든 장의 노트북에서 그대로 씁니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [1]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

사용하는 API: anthropic


In [2]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install sentence-transformers rank-bm25 pypdf
# 9장과 관통 프로젝트 4단계에서 씁니다. 8장의 준비에 평가 함수와 리랭커, 채점 프롬프트를 더했습니다
import csv
import random
import re

import numpy as np

import corpus
import embed

with open(f'{DATA}/문의.csv', encoding='utf-8') as file:
    inquiries = list(csv.DictReader(file))        # 문의 60건. 관통 프로젝트 1, 2단계의 코드가 씁니다
LABELS = ['설비 이상', '품질', '안전', '문서와 데이터 요청']

# 100만 토큰당 단가(달러). 5장과 같습니다(llm.py가 쓰는 Anthropic 모델, 2026-11 확인)
PRICES = {'소형': {'input': 0.10, 'output': 0.50}, '기본': {'input': 2.00, 'output': 10.00},
          '대형': {'input': 4.00, 'output': 20.00}}


def cost(reply, tier):
    """응답 하나의 요금(달러). 입력 토큰과 출력 토큰(thinking 포함)에 각각의 단가를 곱해 더합니다"""
    price = PRICES[tier]
    return (reply.input_tokens * price['input'] + reply.output_tokens * price['output']) / 1_000_000


catalog = corpus.catalog()                               # {문서 id: 메타데이터}. 문서 120건
questions = {item['id']: item for item in corpus.evalset()}   # 평가 질문 62개
chunks, failed = corpus.load_chunks()                    # 문서마다 파싱하고 절 단위 청크로 나눕니다(corpus.py)
searcher = corpus.Search(chunks)                         # 청크마다 임베딩과 BM25 인덱스를 만듭니다. 1~2분 걸립니다


def is_valid(chunk):
    """메타데이터 필터. 폐기된 구판의 청크를 검색 대상에서 뺍니다"""
    return chunk['status'] == '유효'


RAG_PROMPT = """아래 [검색 결과]만 근거로 질문에 답하세요.
- 문장마다 근거가 된 청크의 번호를 [SOP-WB-001#3]처럼 적습니다.
- 검색 결과에 답이 없으면 "검색한 문서에서 찾지 못했습니다"라고만 답합니다.
- 검색 결과끼리 내용이 다르면 날짜가 늦은 문서를 따르고, 내용이 다르다는 것을 함께 적습니다.

[검색 결과]
{context}

[질문]
{question}"""


def rag_prompt(question, hits, with_meta=True):
    """검색한 청크를 번호와 함께 RAG_PROMPT에 넣습니다. with_meta가 참이면 청크마다 상태, 개정, 날짜를 붙입니다"""
    blocks = []
    for hit in hits:
        meta = f" (상태 {hit['status']}, 개정 {hit['revision']}, {hit['date']})" if with_meta else ''
        blocks.append(f"[{hit['chunk_id']}]{meta}\n{hit['text']}")
    return RAG_PROMPT.format(context='\n\n'.join(blocks), question=question)


def docs_of(indices):
    """청크 번호의 순위를 문서 id의 순위로 접습니다. 한 문서의 청크가 여러 번 나오면 처음 나온 순위만 남깁니다"""
    order = []
    for index in indices:
        if chunks[index]['doc_id'] not in order:
            order.append(chunks[index]['doc_id'])
    return order


def first_hit(order, gold):
    """문서 순위에서 정답 문서가 처음 나오는 순위. 없으면 None"""
    return next((rank for rank, doc_id in enumerate(order, 1) if doc_id in gold), None)


def load_reranker():
    """예제 9-2용. 크로스 인코더 리랭커를 PC에서 불러옵니다. 처음 한 번 수백 MB를 내려받습니다"""
    from sentence_transformers import CrossEncoder
    name = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'
    embed.ask_endpoint(name)                     # 모델이 PC에 없고 HF_ENDPOINT가 비어 있으면 주소를 묻습니다
    return CrossEncoder(name)


def rag_answer(question, k=4):
    """8장의 RAG. 유효한 문서에서 청크 k개를 검색해 RAG_PROMPT로 묻고 (응답, 검색한 청크)를 돌려줍니다"""
    hits = searcher.search(question, k=k, where=is_valid)
    return llm.ask(rag_prompt(question, hits), model='소형', max_tokens=2000), hits


JUDGE_PROMPT = """너는 사내 문서 질의응답 시스템의 채점자다. [답]을 아래 두 기준으로 따로 채점한다.
- faithful: [답]의 내용이 모두 [검색 결과]에서 확인되면 true, 검색 결과에 없는 내용이 하나라도 있으면 false.
- correct: [답]이 [기준 답]과 같은 사실을 말하면 true. 기준 답이 "문서에 답이 없다"이면, 찾지 못했다고 답했을 때만 true.
reason에는 판정의 근거를 한 문장으로 쓴다.

[검색 결과]
{context}

[질문]
{question}

[기준 답]
{reference}

[답]
{answer}"""
JUDGE_SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['faithful', 'correct', 'reason'],
                'properties': {'faithful': {'type': 'boolean'}, 'correct': {'type': 'boolean'}, 'reason': {'type': 'string'}}}


print(f'문서 {len(catalog)}건, 청크 {len(chunks)}개, 글을 뽑지 못한 문서 {failed}')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

C:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


임베딩 모델: intfloat/multilingual-e5-small (PC에 저장된 파일을 씁니다)


문서 120건, 청크 487개, 글을 뽑지 못한 문서 ['EHS-010']
소형 모델: claude-haiku-5-5
기본 모델: claude-sonnet-5-5
대형 모델: claude-opus-5-5


준비 완료


오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. 커널을 다시 시작하고 "준비 (1)" 셀에서 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 앞 단계의 코드: 1단계 풀이

In [3]:
SYSTEM = '너는 반도체 후공정 라인의 문의에 답하는 도우미다. 사내 정보를 모르면 지어내지 말고 모른다고 답한다.'


def answer(question, tier='소형', tries=3, send=llm.send, sleep=time.sleep):
    """질문 하나에 답합니다. 돌려주는 dict의 키: question, status('완료', '확인 필요', '실패'),
    answer, input_tokens, output_tokens, cost, reason"""
    result = {'question': question, 'status': '실패', 'answer': '', 'input_tokens': 0,
              'output_tokens': 0, 'cost': 0.0, 'reason': ''}
    for attempt in range(tries):
        try:
            reply = send([{'role': 'user', 'content': question}], system=SYSTEM, model=tier, max_tokens=2000)
        except llm.CallError as error:
            result['reason'] = str(error)
            if not error.retryable:
                return result                         # 요청을 고쳐야 풀리는 오류는 다시 보내지 않습니다
            if attempt < tries - 1:
                sleep(2 ** attempt + random.random())  # 1초, 2초 … 백오프에 지터를 더합니다
            continue
        result.update(answer=reply.text, input_tokens=reply.input_tokens, output_tokens=reply.output_tokens,
                      cost=cost(reply, tier), reason=reply.stop_reason,
                      status='완료' if reply.stop_reason == 'end' else '확인 필요')
        return result
    return result


def mock_send(results):
    """미리 정해 둔 결과를 차례로 돌려주는 mock. 결과가 예외이면 발생시킵니다"""
    queue = list(results)

    def send(messages, **options):
        result = queue.pop(0)
        if isinstance(result, Exception):
            raise result
        return result
    return send


done = llm.Reply(text='완료된 답', input_tokens=50, output_tokens=20, stop_reason='end', model='mock', seconds=0.1)
cut = llm.Reply(text='잘린 답', input_tokens=50, output_tokens=2000, stop_reason='max_tokens', model='mock', seconds=0.1)
busy, missing = llm.CallError('rate_limit', 429, '요청이 많음'), llm.CallError('not_found', 404, '없는 모델')
no_wait = lambda seconds: None                       # 테스트에서는 기다리지 않습니다
print('429 두 번 뒤 성공:', answer('질문', send=mock_send([busy, busy, done]), sleep=no_wait)['status'])
print('429 세 번:', answer('질문', send=mock_send([busy, busy, busy]), sleep=no_wait)['status'])
print('404 한 번(다음 응답은 성공):', answer('질문', send=mock_send([missing, done]), sleep=no_wait)['status'])
cut_result = answer('질문', send=mock_send([cut]), sleep=no_wait)
print('잘린 응답:', cut_result['status'], cut_result)

results = [answer(item['문의']) for item in inquiries if item['번호'] in ('Q31', 'Q33', 'Q35', 'Q36', 'Q38')]
for result in results:
    print(f"\n[{result['status']}] {result['question']}\n  → {result['answer'].strip()[:100]}")
print(f"\n요금 합계 {sum(result['cost'] for result in results):.5f}달러, "
      f"실패와 확인 필요 목록 {[(result['question'], result['reason']) for result in results if result['status'] != '완료']}")

429 두 번 뒤 성공: 완료
429 세 번: 실패
404 한 번(다음 응답은 성공): 실패
잘린 응답: 확인 필요 {'question': '질문', 'status': '확인 필요', 'answer': '잘린 답', 'input_tokens': 50, 'output_tokens': 2000, 'cost': 0.001005, 'reason': 'max_tokens'}



[완료] 와이어 본딩 작업 표준 최신본은 어디에 있나요?
  → 와이어 본딩 작업 표준의 최신본이 어디에 있는지는 제가 알 수 없습니다. 사내 문서 시스템에는 접근할 수 없어서, 위치를 추측해서 안내하면 잘못된 문서를 참조할 수 있습니다.

확

[완료] 몰딩 컴파운드 보관 조건이 적힌 문서를 찾고 있습니다.
  → 몰딩 컴파운드 보관 조건이 적힌 문서는 제가 직접 검색하거나 열어볼 수 없습니다. 사내 문서 시스템에도 접근할 수 없어서 해당 문서의 존재 여부나 내용을 확인해 드릴 수 없습니다.

[완료] 신규 제품 QFN-48의 검사 기준서를 공유해 주세요.
  → 죄송하지만 이 대화에서는 신규 제품 QFN-48의 검사 기준서에 접근할 수 없습니다. 제가 가진 정보에 해당 문서가 없어서, 내용을 임의로 만들어 드리면 실제 라인에서 잘못된 기준

[완료] 3월 품질 회의 회의록을 받을 수 있을까요?
  → 죄송하지만 저는 사내 문서 저장소나 회의록 시스템에 접근할 수 없어서, 3월 품질 회의록을 직접 받아 드리거나 내용을 확인해 드릴 수는 없습니다. 회의록이 있는지, 어떤 내용인지도

[완료] 교대 인수인계 양식 파일이 어디 있는지 알려 주세요.
  → 죄송하지만 사내 파일 서버나 문서 시스템에는 접근할 수 없어서 교대 인수인계 양식이 어디에 있는지 알 수 없습니다. 임의의 경로를 알려드리면 잘못된 정보가 될 수 있어 드리지 않겠

요금 합계 0.00158달러, 실패와 확인 필요 목록 []


## 앞 단계의 코드: 2단계 풀이

In [4]:
import functools

ANSWER_SCHEMA = {'type': 'object', 'additionalProperties': False,
                 'required': ['category', 'urgent', 'needs_document', 'answer'],
                 'properties': {'category': {'type': 'string', 'enum': LABELS}, 'urgent': {'type': 'boolean'},
                                'needs_document': {'type': 'boolean'}, 'answer': {'type': 'string'}}}


def check(data):
    """업무 규칙 검사. 문제가 없으면 None, 있으면 사유 문자열을 돌려줍니다"""
    if data['category'] == '문서와 데이터 요청' and not data['needs_document']:
        return '문서 요청인데 사내 문서가 필요하지 않다고 답함'
    if data['category'] == '안전' and not data['urgent']:
        return '안전 문의인데 긴급이 아니라고 답함'
    return None


def answer_structured(question):
    """1단계의 answer()에 구조화 출력을 더합니다. 결과 dict에 data(파싱한 dict)와 status, reason을 넣습니다"""
    result = answer(question, send=functools.partial(llm.send, schema=ANSWER_SCHEMA))
    result['data'] = None
    if result['status'] != '완료':                   # 잘림과 거절은 스키마에 맞지 않을 수 있으므로 파싱하지 않습니다
        result['status'] = '실패'
        return result
    result['data'] = json.loads(result['answer'])
    problem = check(result['data'])
    result['status'], result['reason'] = ('사람 확인', problem) if problem else ('자동 처리', '')
    return result


for item in [item for item in inquiries if item['번호'] in ('Q21', 'Q24', 'Q31', 'Q43', 'Q49', 'Q56')]:
    result = answer_structured(item['문의'])
    print(f"[{result['status']}] {item['번호']} {item['문의']}\n  → {result['data']}, 사유: {result['reason']}, 요금 {result['cost']:.5f}달러")

[자동 처리] Q21 클린룸 B구역 바닥에 약품이 흘러 있는데 무슨 약품인지 모르겠습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '즉시 해당 구역 출입을 통제하고 주변 작업자를 대피시키세요. 흘러 있는 액체는 임의로 만지거나 닦지 말고, 가능하면 환기 상태를 유지하며 시설 안전 담당자와 생산 관리자에게 바로 보고하세요. 약품 종류는 제가 사내 정보를 알 수 없어 확인해 드릴 수 없으니, 라인 작업 기록이나 해당 공정의 MSDS(물질안전보건자료)를 확인하거나 담당 부서에 문의해 주세요. 누출 대응 절차가 있다면 그 절차를 따르세요.'}, 사유: , 요금 0.00018달러


[자동 처리] Q24 비상구 앞에 자재 카트가 쌓여 통로가 막혔습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '비상구 앞 통로가 막혀 있으면 즉시 위험합니다. 카트를 바로 치우고 통로를 확보해 주세요. 작업자가 있다면 주변 작업을 잠시 멈추고 안내해 주시고, 치우는 중에는 다른 사람이 접근하지 않도록 구역을 표시하세요. 카트를 누가 어디에 두었는지 파악해 재발 방지를 위해 현장 안전 담당자 또는 관리자에게 바로 알려 주시기 바랍니다. 사내 비상구 관련 규정이나 담당 연락처는 제가 알지 못하므로 현장 안전 책임자에게 확인해 주세요.'}, 사유: , 요금 0.00018달러


[자동 처리] Q31 와이어 본딩 작업 표준 최신본은 어디에 있나요?
  → {'category': '문서와 데이터 요청', 'urgent': False, 'needs_document': True, 'answer': "와이어 본딩 작업 표준의 최신본 위치는 제가 사내 문서 저장소 정보를 가지고 있지 않아 확인할 수 없습니다. 추측으로 경로를 안내하지 않겠습니다. 현장 관리자나 공정 기술팀(문서 관리 담당)에게 문의하시거나, 사내 문서 관리 시스템에서 '와이어 본딩 작업 표준'으로 검색하시고 문서의 개정 번호와 승인 일자가 최신인지 확인하시기 바랍니다."}, 사유: , 요금 0.00016달러


[자동 처리] Q43 다이싱 소 D-01 근처에서 냉각수가 바닥으로 새고 있습니다.
  → {'category': '안전', 'urgent': True, 'needs_document': False, 'answer': '냉각수 누수는 전기 설비와 미끄러짐 위험이 있으므로 즉시 조치가 필요합니다. 우선 D-01 장비의 전원을 차단하고(안전 절차에 따라 비상 정지 버튼 사용), 누수 구역 주변 작업자를 통제하며 바닥을 정리해 미끄럼을 방지하세요. 누수 원인 확인과 복구는 담당 설비 엔지니어에게 즉시 연락해 진행하고, 사내 비상 연락망과 누수 처리 절차는 제가 알지 못하므로 현장 안전 담당자나 매뉴얼을 확인해 주세요.'}, 사유: , 요금 0.00018달러


[자동 처리] Q49 마킹 불량 제품이 출하 박스에 섞였는지 확인할 출하 기록이 필요합니다.
  → {'category': '문서와 데이터 요청', 'urgent': True, 'needs_document': True, 'answer': '마킹 불량 제품이 출하 박스에 섞였는지 확인하려면 해당 출하 박스의 출하 기록이 필요합니다. 출하 기록에서 박스 번호, 로트 번호, 시리얼 번호, 출하 일시, 검사 결과를 확인하시면 됩니다. 다만 저는 사내 시스템이나 실제 출하 데이터에 접근할 수 없어 기록의 내용을 직접 확인해 드릴 수 없습니다. 출하 기록은 MES나 출하 관리 시스템에서 조회하시거나 담당 부서에 요청하시기 바랍니다. 불량 제품이 출하되었을 가능성이 있으니 출하 보류와 해당 박스 재검사도 함께 검토하시는 것을 권장합니다.'}, 사유: , 요금 0.00021달러


[자동 처리] Q56 본더 한 대가 또 멈췄습니다.
  → {'category': '설비 이상', 'urgent': True, 'needs_document': False, 'answer': '본더 정지는 라인 생산에 직접 영향을 주므로 긴급하게 처리해야 합니다. 우선 설비 알람 메시지와 정지 시점, 직전 작업 조건을 확인해 주세요. 현재 사내 정지 이력이나 정비 기록은 제가 알 수 없으니, 해당 본더 담당 설비 엔지니어에게 즉시 연락해 점검을 요청하시기 바랍니다. 반복 정지라면 이전 정지 이력과 공통 원인 여부도 함께 확인하는 것이 좋습니다.'}, 사유: , 요금 0.00016달러


## 앞 단계의 코드: 3단계 풀이

In [5]:
def answer_with_documents(question, k=4):
    """사내 문서를 검색해 근거를 붙여 답합니다. 1단계 answer()의 결과에 hits와 sources를 더합니다"""
    hits = searcher.search(question, k=k, where=is_valid)
    result = answer(rag_prompt(question, hits))
    result['question'] = question
    result['hits'] = [hit['chunk_id'] for hit in hits]
    result['sources'] = sorted(set(re.findall(r'[A-Z]+-[\w-]+#\d+', result['answer'])))
    if result['status'] == '완료' and '찾지 못했습니다' not in result['answer']:
        if not result['sources']:
            result['status'], result['reason'] = '확인 필요', '출처 없음'
        elif set(result['sources']) - set(result['hits']):
            result['status'], result['reason'] = '확인 필요', '검색하지 않은 출처'
    return result


for qid in ['E06', 'E24', 'E28', 'E41', 'E55']:
    result = answer_with_documents(questions[qid]['question'])
    print(f"[{result['status']}] {qid} {result['question']} (정답: {questions[qid]['answer']})")
    print(f"  → {result['answer'].strip().replace(chr(10), ' ')[:160]}")
    print(f"  검색 {result['hits']}, 출처 {result['sources']}, 사유 {result['reason']}")

[완료] E06 패키지 테스트에서 로트 수율이 몇 % 미만이면 로트를 홀드하나요? (정답: 97% 미만이면 홀드한다.)
  → 패키지 테스트에서 로트 수율이 **97% 미만**이면 로트를 홀드하고 테스트기술팀과 품질보증팀에 알립니다 [SOP-PT-001#4].
  검색 ['SOP-PT-001#4', 'QIR-2026-015#2', 'EDU-006#3', 'MTG-20260601#3'], 출처 ['SOP-PT-001#4'], 사유 end


[완료] E24 D-03 블레이드 때문에 생긴 칩핑 불량 때 블레이드의 누적 절단 거리는 얼마였나요? (정답: 11.5km.)
  → D-03 블레이드의 누적 절단 거리는 **11.5km**였습니다 [QIR-2025-044#3].  당시 교체 기준은 12km였고 블레이드는 기준에 이르지 않았습니다 [QIR-2025-044#3]. 이후 교체 기준은 10km로 낮아졌습니다 [QIR-2025-044#5][MAN-DS-001
  검색 ['QIR-2025-044#3', 'MAN-DS-001#3', 'QIR-2025-044#5', 'MTG-20260119#3'], 출처 ['MAN-DS-001#3', 'QIR-2025-044#3', 'QIR-2025-044#5'], 사유 end


[완료] E28 몰딩 컴파운드는 냉장 창고에서 꺼내 최소 몇 시간 해동하나요? (정답: 최소 6시간(SOP-MD-002 2차 개정).)
  → 냉장 창고에서 꺼낸 몰딩 컴파운드는 포장을 뜯지 않은 채 상온에서 최소 6시간 해동합니다. [SOP-MD-002#3]
  검색 ['SOP-MD-002#4', 'SOP-MD-002#3', 'SOP-MD-002#2', 'SOP-MD-002#1'], 출처 ['SOP-MD-002#3'], 사유 end


[완료] E41 QX48의 몰딩 트랜스퍼 시간은 지금 몇 초인가요? (정답: 14초. 2026-11-16 회의에서 11월 23일부터 14초로 바꾸기로 했다. 작업 표준(SOP-MD-001 4차 개정)은 아직 12초로 남아 있다.)
  → 검색 결과 기준으로 가장 최근 문서는 QX48 트랜스퍼 시간을 **14초**로 정하고 있습니다. 다만 적용일은 **11월 23일부터**입니다 [MTG-20261116#4].  검색 결과끼리 내용이 다릅니다. 2025년 문서는 12초를 기준으로 설명하고 있습니다 [QIR-2025-031#
  검색 ['QIR-2025-031#5', 'QIR-2025-031#4', 'QIR-2026-019#4', 'MTG-20261116#4'], 출처 ['MTG-20261116#4', 'QIR-2025-031#4', 'QIR-2025-031#5', 'QIR-2026-019#4'], 사유 end


[완료] E55 QX64의 몰딩 금형 온도는 몇 도인가요? (정답: 문서에 답이 없다. QX64의 공정 조건은 아직 작업 표준에 추가되지 않았다.)
  → 검색한 문서에서 QX64의 몰딩 금형 온도는 찾지 못했습니다.  참고로 관련된 내용은 다음과 같습니다. - 몰딩 금형 클리닝 SOP에 금형 온도 175℃가 적혀 있으나, 이 문서는 특정 제품을 명시하지 않은 일반 절차입니다. 따라서 QX64의 몰딩 온도로 확정할 수 없습니다 [SOP-M
  검색 ['MTG-20260601#3', 'SOP-MD-003#4', 'EDU-013#1', 'EDU-013#2'], 출처 ['MTG-20260601#3', 'SOP-MD-003#4'], 사유 end


## 실습 P4. 평가셋으로 측정하고 고치기

교재의 절: "관통 프로젝트 4단계"

`evaluate(qids, k)`를 작성합니다. 질문마다 3단계의 `answer_with_documents(question, k)`로 답하고, 검색한 청크에 정답 문서가 있었는지(`found`)와 답이 맞았는지(`correct`)를 기록한 행의 목록을 돌려줍니다. 답이 맞았는지는 핵심어(`key`)가 모두 답에 들어 있는지로, 답이 없는 질문은 "찾지 못했습니다"가 답에 있는지로 정합니다. `cells()`는 행 목록을 네 칸(답 없음 질문은 따로)으로 세어 돌려줍니다. k=4와 k=8의 네 칸 건수와 요금이 출력되면 끝납니다. 이 셀은 API를 24번 호출합니다.

완료 기준

- k마다 네 칸과 답 없음 질문의 건수, 요금 합계가 출력됩니다.
- 실패한 질문의 id가 칸별로 출력됩니다.

<details><summary>힌트 보기</summary>

청크 번호의 `#` 앞이 문서 id입니다(`'SOP-WB-001#4'.split('#')[0]`). 3단계의 결과 dict에는 `hits`(검색한 청크 번호)와 `cost`(1단계에서 계산한 요금)가 있습니다.

</details>

In [6]:
EVAL_IDS = ['E06', 'E13', 'E19', 'E23', 'E31', 'E37', 'E40', 'E41', 'E44', 'E53', 'E55', 'E58']


def evaluate(qids, k=4):
    """질문마다 {'id', 'found', 'correct', 'cost'}를 담은 행의 목록. 답이 없는 질문의 found는 None입니다"""
    rows = []
    for qid in qids:
        item = questions[qid]
        result = answer_with_documents(item['question'], k=k)
        searched = {hit.split('#')[0] for hit in result['hits']}
        found = bool(searched & set(item['docs'])) if item['docs'] else None
        if item['key']:
            correct = all(key in result['answer'] for key in item['key'])
        else:
            correct = '찾지 못했습니다' in result['answer']
        rows.append({'id': qid, 'found': found, 'correct': correct, 'cost': result['cost']})
    return rows


def cells(rows):
    """행을 네 칸과 답 없음으로 나눠 {칸 이름: [질문 id]}로 돌려줍니다"""
    names = {(True, True): '찾고 맞음', (True, False): '찾고 틀림(생성 실패)',
             (False, True): '못 찾고 맞음', (False, False): '못 찾고 틀림(검색 실패)'}
    grouped = {name: [] for name in names.values()}
    grouped.update({'답 없음, 맞음': [], '답 없음, 틀림': []})
    for row in rows:
        if row['found'] is None:
            grouped['답 없음, 맞음' if row['correct'] else '답 없음, 틀림'].append(row['id'])
        else:
            grouped[names[(row['found'], row['correct'])]].append(row['id'])
    return grouped


for k in [4, 8]:
    rows = evaluate(EVAL_IDS, k)
    print(f"k={k}: 요금 {sum(row['cost'] for row in rows):.4f}달러")
    for name, ids in cells(rows).items():
        print(f'  {name} {len(ids)}건: {ids}')

k=4: 요금 0.0045달러
  찾고 맞음 7건: ['E06', 'E13', 'E19', 'E31', 'E37', 'E41', 'E53']
  찾고 틀림(생성 실패) 1건: ['E23']
  못 찾고 맞음 0건: []
  못 찾고 틀림(검색 실패) 2건: ['E40', 'E44']
  답 없음, 맞음 2건: ['E55', 'E58']
  답 없음, 틀림 0건: []


k=8: 요금 0.0069달러
  찾고 맞음 8건: ['E06', 'E13', 'E19', 'E23', 'E31', 'E37', 'E41', 'E53']
  찾고 틀림(생성 실패) 0건: []
  못 찾고 맞음 0건: []
  못 찾고 틀림(검색 실패) 2건: ['E40', 'E44']
  답 없음, 맞음 2건: ['E55', 'E58']
  답 없음, 틀림 0건: []
